# Time-Use Profiles Evaluation

Use one notebook for both the baseline donor method and the archetype-assignment method.

Set `METHOD = "baseline"` or `METHOD = "archetype"` in the config cell.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.timeuse_profiles import (
    build_evaluation_bundle,
    build_match_level_summary,
    build_respondent_profiles,
    build_weak_group_summary,
    compare_distribution,
    derive_tus_matching_features,
    filter_day_type_inputs,
    prepare_timeuse_data,
)


In [ ]:
DAY_TYPE = "weekday"
METHOD = "baseline"   # baseline or archetype
PROC_TIME_USE_DIR = PROJECT_ROOT / "data" / "processed" / "timeuse_profiles"
METHOD_TAG = None if METHOD == "baseline" else METHOD
OUT_DIR = PROC_TIME_USE_DIR / ("evaluation" if METHOD_TAG is None else f"evaluation_{METHOD_TAG}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DONOR_ASSIGNMENTS_PATH = (
    PROC_TIME_USE_DIR / f"synthetic_to_tus_donors_{DAY_TYPE}.csv"
    if METHOD_TAG is None
    else PROC_TIME_USE_DIR / f"synthetic_to_tus_donors_{DAY_TYPE}_{METHOD_TAG}.csv"
)
DISTRICT_PROFILES_PATH = (
    PROC_TIME_USE_DIR / f"district_hourly_profiles_{DAY_TYPE}.csv"
    if METHOD_TAG is None
    else PROC_TIME_USE_DIR / f"district_hourly_profiles_{DAY_TYPE}_{METHOD_TAG}.csv"
)
WEAK_AGE_GROUPS = ["75_plus", "18_24"]


In [ ]:
prepared = prepare_timeuse_data()
day_resp, day_ep = filter_day_type_inputs(prepared.tus_resp_raw, prepared.tus_ep_raw, day_type=DAY_TYPE)
tus_people = derive_tus_matching_features(day_resp)
respondent_profiles = build_respondent_profiles(day_ep, config=prepared.config)

donors = pd.read_csv(DONOR_ASSIGNMENTS_PATH)
district_profiles = pd.read_csv(DISTRICT_PROFILES_PATH)

syn_people = prepared.syn_people_all.copy()
syn_people["person_id"] = syn_people["person_id"].astype(str)
tus_people["respondent_id"] = tus_people["respondent_id"].astype(str)
donors["person_id"] = donors["person_id"].astype(str)
if "respondent_id" in donors.columns:
    donors["respondent_id"] = donors["respondent_id"].astype(str)

assigned = syn_people.merge(donors, on="person_id", how="left", validate="1:1")
assigned = assigned.merge(
    tus_people.add_prefix("tus_"),
    left_on="respondent_id",
    right_on="tus_respondent_id",
    how="left",
)

tus_supported_age_bands = sorted(tus_people["age_band"].astype(str).dropna().unique().tolist())
assigned_eval = assigned.loc[assigned["age_band"].astype(str).isin(tus_supported_age_bands)].copy()
tus_eval = tus_people.loc[tus_people["age_band"].astype(str).isin(tus_supported_age_bands)].copy()

bundle = build_evaluation_bundle(
    syn_raw=prepared.syn_raw,
    assigned_eval=assigned_eval,
    tus_eval=tus_eval,
    district_profiles=district_profiles,
    respondent_profiles=respondent_profiles,
    household_id_col=prepared.config.household_id_eval_col,
)

display(bundle.match_summary)
display(bundle.household_summary)
display(bundle.district_summary.head())
display(bundle.subgroup_jsd_summary)
display(bundle.subgroup_hourly_profile_errors.head())


In [ ]:
bundle.match_summary.to_csv(OUT_DIR / f"match_summary_{DAY_TYPE}.csv", index=False)
bundle.match_level_summary.to_csv(OUT_DIR / f"match_levels_{DAY_TYPE}.csv", index=False)
bundle.household_summary.to_csv(OUT_DIR / f"household_summary_{DAY_TYPE}.csv", index=False)
bundle.district_peak_home.to_csv(OUT_DIR / f"district_peak_home_{DAY_TYPE}.csv", index=False)
bundle.district_mean_profiles.to_csv(OUT_DIR / f"district_mean_profiles_{DAY_TYPE}.csv", index=False)
bundle.district_summary.to_csv(OUT_DIR / f"district_summary_{DAY_TYPE}.csv", index=False)
bundle.subgroup_jsd_summary.to_csv(OUT_DIR / f"academic_jsd_summary_{DAY_TYPE}.csv", index=False)
bundle.subgroup_hourly_profile_errors.to_csv(OUT_DIR / f"subgroup_hourly_profile_errors_{DAY_TYPE}.csv", index=False)

weak_group_summary = build_weak_group_summary(
    assigned_eval,
    tus_eval,
    weak_age_groups=WEAK_AGE_GROUPS,
)
weak_group_summary.to_csv(OUT_DIR / f"weak_age_group_summary_{DAY_TYPE}.csv", index=False)
display(weak_group_summary)
